# 🤖 Agentic RAG System — Financial QA
### Architecture: Planner → Router → Specialist Agents (Text / Tablular / Metadata) → Critic → Answer Synthesizer → Evaluation metrics → Evaluation Report

## 📦 Step 1 — Install Dependencies

In [1]:
!pip install pymupdf pandas tqdm sentence-transformers faiss-cpu rank_bm25 \
             langchain langchain-community langchain-text-splitters langchain-groq torch -q

## 📚 Step 2 — Imports

In [2]:
import json, re, time
from tqdm import tqdm
from io import StringIO
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

import pandas as pd
import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

print("✅ All imports successful")

✅ All imports successful


## 🔑 Step 3 — LLM & Embedding Model Setup

In [3]:
import os
# ── Groq LLM ──────────────────────────────────────────────────────────────
GROQ_API_KEY = os.environ["GROQ_API_KEY"]  # set this in your environment / .env — never hardcode keys

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0
)

# ── Embedding & Reranker ────────────────────────────────────────────────────
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker        = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def embed_texts(texts):
    return embedding_model.encode(texts, normalize_embeddings=True)

print("✅ LLM, embeddings, and reranker ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ LLM, embeddings, and reranker ready


## 📂 Step 4 — Data Loading & Preprocessing

In [4]:
# ── Loaders ─────────────────────────────────────────────────────────────────
def load_jsonl(file_path):
    data = []
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

def clean_text(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def parse_table(table_str):
    """Parse raw table string → human-readable markdown table."""
    try:
        df = pd.read_csv(StringIO(table_str))
        return df.to_markdown(index=False)          # markdown = better for LLMs
    except Exception:
        return str(table_str)

def prepare_dataset(data, max_samples=500):
    processed = []
    for item in tqdm(data[:max_samples], desc="Preprocessing"):
        try:
            full_text = " ".join(filter(None, [
                clean_text(item.get("pre_text", "")),
                clean_text(item.get("context", "")),
                clean_text(item.get("post_text", ""))
            ])).strip()

            processed.append({
                "question": clean_text(item.get("question", "")),
                "text":     full_text,
                "table":    parse_table(item.get("table", "")),
                "answer":   clean_text(item.get("original_answer", "")),
                "metadata": {
                    "company": item.get("company_name"),
                    "year":    item.get("report_year"),
                    "sector":  item.get("company_sector"),
                }
            })
        except Exception as e:
            print(f"⚠️  Skipping sample: {e}")
    return processed


data         = load_jsonl("/content/turn_0.jsonl")
processed_data = prepare_dataset(data, max_samples=500)
print(f"\n✅ Loaded {len(processed_data)} samples")

Preprocessing: 100%|██████████| 500/500 [00:04<00:00, 117.65it/s]


✅ Loaded 500 samples


## 🗂️ Step 5 — Smart Chunking & Separate Indexes
> Three separate FAISS indexes: **text**, **table**, **metadata** — one per specialist agent.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

# ── Chunk all three modalities ───────────────────────────────────────────────
text_chunks     = []   # for Text Agent
table_chunks    = []   # for Table Agent
metadata_chunks = []   # for Metadata Agent

for item in processed_data:
    base_meta = item["metadata"]

    # TEXT chunks
    for c in splitter.split_text(item["text"]):
        text_chunks.append({"content": c, "type": "text", "metadata": base_meta})

    # TABLE chunks (kept whole — no splitting breaks tabular structure)
    if item["table"] and len(item["table"].strip()) > 10:
        table_chunks.append({"content": item["table"], "type": "table", "metadata": base_meta})

    # METADATA chunks (structured string for filtering/lookup)
    meta_str = (
        f"Company: {base_meta.get('company','N/A')} | "
        f"Year: {base_meta.get('year','N/A')} | "
        f"Sector: {base_meta.get('sector','N/A')}"
    )
    metadata_chunks.append({"content": meta_str, "type": "metadata", "metadata": base_meta})

print(f"Text chunks:     {len(text_chunks)}")
print(f"Table chunks:    {len(table_chunks)}")
print(f"Metadata chunks: {len(metadata_chunks)}")

# ── Build FAISS index helper ─────────────────────────────────────────────────
def build_faiss_index(chunks):
    texts = [c["content"] for c in chunks]
    embs  = embed_texts(texts)
    dim   = embs.shape[1]
    idx   = faiss.IndexFlatIP(dim)
    idx.add(np.array(embs))
    return idx, texts

text_index,     text_texts     = build_faiss_index(text_chunks)
table_index,    table_texts    = build_faiss_index(table_chunks)
metadata_index, metadata_texts = build_faiss_index(metadata_chunks)

# ── BM25 for each modality ───────────────────────────────────────────────────
bm25_text     = BM25Okapi([t.split() for t in text_texts])
bm25_table    = BM25Okapi([t.split() for t in table_texts])
bm25_metadata = BM25Okapi([t.split() for t in metadata_texts])

print("\n✅ All three indexes built")

Text chunks:     10439
Table chunks:    500
Metadata chunks: 500

✅ All three indexes built


## 🔧 Step 6 — Shared Retrieval Utilities
> Dense, Sparse, RRF Fusion, Reranker — reusable by all agents.

In [6]:
# ── Dense Retrieval ─────────────────────────────────────────────────────────
def dense_retrieve(query, faiss_index, top_k=10):
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores, indices = faiss_index.search(q_emb, top_k)
    return [{"doc_id": int(indices[0][i]), "score": float(scores[0][i])} for i in range(len(indices[0]))]

# ── Sparse Retrieval ─────────────────────────────────────────────────────────
def sparse_retrieve(query, bm25_index, top_k=10):
    scores      = bm25_index.get_scores(query.split())
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"doc_id": int(idx), "score": float(scores[idx])} for idx in top_indices]

# ── RRF Fusion ───────────────────────────────────────────────────────────────
def rrf_fusion(dense_results, sparse_results, k=60):
    fusion = {}
    for rank, item in enumerate(dense_results):
        fusion[item["doc_id"]] = fusion.get(item["doc_id"], 0) + 1 / (k + rank)
    for rank, item in enumerate(sparse_results):
        fusion[item["doc_id"]] = fusion.get(item["doc_id"], 0) + 1 / (k + rank)
    return sorted(fusion.items(), key=lambda x: x[1], reverse=True)

# ── Cross-Encoder Reranker ───────────────────────────────────────────────────
def rerank(query, candidate_docs):
    if not candidate_docs:
        return []
    pairs      = [(query, doc["content"]) for doc in candidate_docs]
    raw_scores = reranker.predict(pairs)
    norm       = torch.sigmoid(torch.tensor(raw_scores)).numpy()

    for i, doc in enumerate(candidate_docs):
        s = float(norm[i])
        q = query.lower()
        # numeric boost
        if any(w in q for w in ["growth", "increase", "decrease", "percentage"]):
            if "%" in doc["content"] or "percent" in doc["content"]:
                s += 0.15
            elif "growth" in doc["content"] and "%" not in doc["content"]:
                s -= 0.05
        # table boost
        if doc.get("type") == "table":
            s += 0.10
        doc["rerank_score"] = s

    return sorted(candidate_docs, key=lambda x: x["rerank_score"], reverse=True)

# ── HyDE ─────────────────────────────────────────────────────────────────────
def hyde_generate(query):
    return (
        f"The financial report discusses detailed numerical values including revenue, "
        f"net income, percentage growth, and year-over-year comparisons. "
        f"Specifically related to: {query}. "
        f"The answer likely involves extracting values from tables and analyzing trends."
    )

print("✅ Retrieval utilities ready")

✅ Retrieval utilities ready


## 🧠 Step 7 — Agent State
> Shared state object passed through the entire agent pipeline.

In [7]:
@dataclass
class AgentState:
    """Shared state object flowing through the entire agentic pipeline."""
    query:           str  = ""
    query_type:      str  = "general"          # numeric | financial | metadata | general
    rewritten_query: str  = ""
    plan:            List[str]  = field(default_factory=list)   # agent execution plan
    agents_to_call:  List[str]  = field(default_factory=list)   # decided by Router
    text_results:    List[Dict] = field(default_factory=list)
    table_results:   List[Dict] = field(default_factory=list)
    metadata_results:List[Dict] = field(default_factory=list)
    merged_context:  str  = ""
    raw_answer:      str  = ""
    critic_feedback: str  = ""
    final_answer:    str  = ""
    iterations:      int  = 0
    MAX_ITER:        int  = 2

print("✅ AgentState defined")

✅ AgentState defined


## 🗺️ Step 8 — Planner Agent
> Analyses the query, rewrites it, detects type, and creates an execution plan.

In [8]:
def planner_agent(state: AgentState) -> AgentState:
    """
    PLANNER AGENT
    -------------
    Responsibilities:
    1. Classify query type (numeric / financial / metadata / general)
    2. Rewrite query for better semantic retrieval
    3. Produce a step-by-step execution plan
    4. Decide which specialist agents are needed
    """
    print("\n🗺️  [PLANNER] Analysing query...")

    query = state.query.lower().strip()

    # ── 1. Classify ──────────────────────────────────────────────────────────
    if any(w in query for w in ["growth", "increase", "decrease", "percentage", "change", "ratio"]):
        state.query_type = "numeric"
    elif any(w in query for w in ["revenue", "income", "profit", "loss", "earnings", "sales", "cost"]):
        state.query_type = "financial"
    elif any(w in query for w in ["company", "sector", "year", "industry", "who", "when", "which"]):
        state.query_type = "metadata"
    else:
        state.query_type = "general"

    # ── 2. Rewrite ────────────────────────────────────────────────────────────
    rewrite_map = {
        "numeric":   " percentage increase year over year revenue growth financial statement",
        "financial": " income statement revenue total revenue financial report",
        "metadata":  " company name sector year industry classification",
        "general":   " financial data business performance annual report",
    }
    state.rewritten_query = state.query + rewrite_map.get(state.query_type, "")

    # ── 3. Plan ───────────────────────────────────────────────────────────────
    plan_map = {
        "numeric":   ["TextAgent", "TableAgent", "Critic", "Synthesizer"],
        "financial": ["TextAgent", "TableAgent", "MetadataAgent", "Critic", "Synthesizer"],
        "metadata":  ["MetadataAgent", "TextAgent", "Critic", "Synthesizer"],
        "general":   ["TextAgent", "MetadataAgent", "Critic", "Synthesizer"],
    }
    state.plan            = plan_map.get(state.query_type, plan_map["general"])
    state.agents_to_call  = [a for a in state.plan if a not in ("Critic", "Synthesizer")]

    print(f"   Query type  : {state.query_type}")
    print(f"   Rewritten   : {state.rewritten_query[:80]}...")
    print(f"   Execution plan: {state.plan}")

    return state

print("✅ Planner Agent ready")

✅ Planner Agent ready


## 🔀 Step 9 — Router Agent
> Uses the LLM to decide which specialist agents to activate, then dispatches.

In [9]:
def router_agent(state: AgentState) -> AgentState:
    """
    ROUTER AGENT
    ------------
    Uses the LLM to validate/override the Planner's agent selection.
    Acts as a smart dispatcher.
    """
    print("\n🔀  [ROUTER] Deciding agent routing...")

    routing_prompt = f"""You are a routing agent for a financial QA system.
Given the query below, decide which specialist agents to call.

Available agents:
- TextAgent      : handles narrative/paragraph financial text
- TableAgent     : handles numerical tables, figures, and percentages
- MetadataAgent  : handles company name, sector, year, and classification queries

Query: \"{state.query}\"
Query type already detected: {state.query_type}
Planner's suggested agents: {state.agents_to_call}

Respond with ONLY a comma-separated list of agent names to call.
Example: TextAgent, TableAgent
"""

    try:
        response = llm.invoke(routing_prompt)
        raw      = response.content.strip()
        valid    = {"TextAgent", "TableAgent", "MetadataAgent"}
        parsed   = [a.strip() for a in raw.split(",") if a.strip() in valid]

        if parsed:
            state.agents_to_call = parsed
            print(f"   Router decision (LLM): {state.agents_to_call}")
        else:
            print(f"   Router kept Planner's plan: {state.agents_to_call}")

    except Exception as e:
        print(f"   ⚠️  Router LLM failed ({e}), keeping Planner plan.")

    return state

print("✅ Router Agent ready")

✅ Router Agent ready


## 📄 Step 10 — Text Agent
> Specialist for narrative / paragraph text chunks. Uses Dense + Sparse + RRF + Reranker.

In [10]:
def text_agent(state: AgentState) -> AgentState:
    """
    TEXT AGENT
    ----------
    Retrieves relevant narrative text chunks.
    Pipeline: HyDE → Dense(FAISS/BGE) + Sparse(BM25) → RRF Fusion → Rerank
    """
    print("\n📄  [TEXT AGENT] Retrieving text chunks...")

    hyde_query = hyde_generate(state.query)

    dense_res  = dense_retrieve(hyde_query,              text_index, top_k=12)
    sparse_res = sparse_retrieve(state.rewritten_query,  bm25_text,  top_k=12)
    fused      = rrf_fusion(dense_res, sparse_res)

    candidates = []
    for doc_id, score in fused[:12]:
        doc = text_chunks[doc_id]
        candidates.append({
            "content":    doc["content"],
            "type":       doc["type"],
            "metadata":   doc["metadata"],
            "base_score": score,
        })

    state.text_results = rerank(state.query, candidates)[:5]

    print(f"   Retrieved {len(state.text_results)} text chunks")
    for i, r in enumerate(state.text_results):
        print(f"   [{i}] score={r['rerank_score']:.4f} | {r['content'][:80]}...")

    return state

print("✅ Text Agent ready")

✅ Text Agent ready


## 📊 Step 11 — Table Agent
> Specialist for numerical tables. Applies extra numeric/percentage boosting.

In [11]:
def table_agent(state: AgentState) -> AgentState:
    """
    TABLE AGENT
    -----------
    Retrieves relevant table chunks.
    Extra boost for chunks containing '%' or numeric patterns (financial focus).
    """
    print("\n📊  [TABLE AGENT] Retrieving table chunks...")

    if not table_chunks:
        print("   No table chunks available.")
        return state

    dense_res  = dense_retrieve(state.rewritten_query, table_index, top_k=10)
    sparse_res = sparse_retrieve(state.rewritten_query, bm25_table,  top_k=10)
    fused      = rrf_fusion(dense_res, sparse_res)

    candidates = []
    for doc_id, score in fused[:10]:
        doc = table_chunks[doc_id]
        candidates.append({
            "content":    doc["content"],
            "type":       "table",
            "metadata":   doc["metadata"],
            "base_score": score,
        })

    reranked = rerank(state.query, candidates)

    # ── Extra numeric pattern boost (post-rerank) ────────────────────────────
    numeric_pattern = re.compile(r"\d+\.?\d*\s*%|\$\s*\d+|\d{4,}")
    for doc in reranked:
        if numeric_pattern.search(doc["content"]):
            doc["rerank_score"] += 0.10
    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)

    state.table_results = reranked[:5]

    print(f"   Retrieved {len(state.table_results)} table chunks")
    for i, r in enumerate(state.table_results):
        print(f"   [{i}] score={r['rerank_score']:.4f} | {r['content'][:80]}...")

    return state

print("✅ Table Agent ready")

✅ Table Agent ready


## 🏷️ Step 12 — Metadata Agent
> Specialist for company/sector/year lookups. Also builds a structured metadata summary.

In [12]:
def metadata_agent(state: AgentState) -> AgentState:
    """
    METADATA AGENT
    --------------
    Retrieves metadata chunks (company, year, sector).
    Also builds a compact structured metadata summary for the synthesizer.
    """
    print("\n🏷️   [METADATA AGENT] Retrieving metadata...")

    dense_res  = dense_retrieve(state.query,    metadata_index, top_k=8)
    sparse_res = sparse_retrieve(state.query,   bm25_metadata,  top_k=8)
    fused      = rrf_fusion(dense_res, sparse_res)

    candidates = []
    for doc_id, score in fused[:8]:
        doc = metadata_chunks[doc_id]
        candidates.append({
            "content":    doc["content"],
            "type":       "metadata",
            "metadata":   doc["metadata"],
            "base_score": score,
        })

    state.metadata_results = rerank(state.query, candidates)[:4]

    # ── Build a compact unique metadata summary ──────────────────────────────
    seen = set()
    summary_lines = []
    for doc in state.metadata_results:
        key = doc["content"]
        if key not in seen:
            seen.add(key)
            summary_lines.append(f"• {doc['content']}")

    print(f"   Retrieved {len(state.metadata_results)} metadata chunks")
    for line in summary_lines:
        print(f"   {line}")

    return state

print("✅ Metadata Agent ready")

✅ Metadata Agent ready


## 🔗 Step 13 — Context Merger
> Merges outputs from all specialist agents into one structured context for the LLM.

In [13]:
def merge_context(state: AgentState) -> AgentState:
    """
    CONTEXT MERGER
    --------------
    Assembles the final context string from all active agents' results.
    Sections are clearly labelled for the LLM.
    """
    print("\n🔗  [MERGER] Building merged context...")

    sections = []

    if state.text_results:
        sections.append("=== NARRATIVE TEXT ===")
        for i, doc in enumerate(state.text_results):
            sections.append(f"[Text {i+1}]\n{doc['content']}")

    if state.table_results:
        sections.append("\n=== TABLES & NUMERICAL DATA ===")
        for i, doc in enumerate(state.table_results):
            sections.append(f"[Table {i+1}]\n{doc['content']}")

    if state.metadata_results:
        sections.append("\n=== COMPANY METADATA ===")
        for i, doc in enumerate(state.metadata_results):
            sections.append(f"[Meta {i+1}]\n{doc['content']}")

    state.merged_context = "\n\n".join(sections)

    print(f"   Context length: {len(state.merged_context)} chars")
    return state

print("✅ Context Merger ready")

✅ Context Merger ready


## ✍️ Step 14 — Answer Synthesizer
> Calls the LLM with the merged context to produce an initial answer.

In [14]:
def synthesizer_agent(state: AgentState) -> AgentState:
    """
    SYNTHESIZER AGENT
    -----------------
    Sends merged context + query to the LLM and generates the raw answer.
    """
    print("\n✍️   [SYNTHESIZER] Generating answer...")

    prompt = f"""You are a financial analyst assistant.
Answer the question STRICTLY based on the provided context.

Instructions:
- Prioritise numerical data and percentages if present.
- Use table data when the question involves figures or growth.
- Use metadata to ground the answer (company, year, sector).
- If the answer is not found in the context, say "Not found in provided context."
- Be concise and precise.

Context:
{state.merged_context}

Question:
{state.query}

Answer:"""

    response         = llm.invoke(prompt)
    state.raw_answer = response.content.strip()

    print(f"   Raw answer: {state.raw_answer[:200]}...")
    return state

print("✅ Synthesizer Agent ready")

✅ Synthesizer Agent ready


## 🔍 Step 15 — Critic Agent
> Reviews the raw answer for hallucinations, missing data, or poor grounding. Triggers re-retrieval if needed.

In [15]:
def critic_agent(state: AgentState) -> AgentState:
    """
    CRITIC AGENT (Fixed)
    --------------------
    Fix 1: Never mutate state.query — use a separate refinement_hint field.
    Fix 2: Track best_answer across iterations; never discard a good answer.
    """
    print("\n🔍  [CRITIC] Evaluating answer quality...")

    # ── Keep best answer seen so far ─────────────────────────────────────────
    if not hasattr(state, 'best_answer'):
        state.best_answer = state.raw_answer   # store first answer

    NOT_FOUND_PHRASES = ["not found", "cannot be determined", "no information"]
    current_is_empty  = any(p in state.raw_answer.lower() for p in NOT_FOUND_PHRASES)
    best_is_empty     = any(p in state.best_answer.lower() for p in NOT_FOUND_PHRASES)

    if not current_is_empty:
        state.best_answer = state.raw_answer   # upgrade best answer

    critic_prompt = f"""You are a strict QA critic for a financial question-answering system.

Question: {state.query}
Generated Answer: {state.raw_answer}
Context (excerpt): {state.merged_context[:1500]}

Evaluate on three criteria:
1. GROUNDED  — Is every claim in the answer supported by the context? (Yes/No)
2. COMPLETE  — Does the answer fully resolve the question? (Yes/No)
3. NUMERICAL — If numbers were available in context, were they used? (Yes/No/NA)

Verdict rules:
- PASS   : answer is grounded AND complete
- REFINE : answer is incomplete but context has more info — give a SHORT (max 8 words) search hint
- FAIL   : answer contradicts context or is hallucinated

Respond EXACTLY in this format:
GROUNDED: Yes/No
COMPLETE: Yes/No
NUMERICAL: Yes/No/NA
VERDICT: PASS/REFINE/FAIL
HINT: <max 8 words, the missing info to search for>"""

    try:
        resp                 = llm.invoke(critic_prompt)
        state.critic_feedback = resp.content.strip()
        print(f"   Critic:\n{state.critic_feedback}")

        verdict_m = re.search(r"VERDICT:\s*(PASS|REFINE|FAIL)", state.critic_feedback)
        hint_m    = re.search(r"HINT:\s*(.+)",                   state.critic_feedback)
        verdict   = verdict_m.group(1) if verdict_m else "PASS"
        hint      = hint_m.group(1).strip()[:80] if hint_m else ""

        if verdict == "REFINE" and state.iterations < state.MAX_ITER:
            print(f"   ⚠️  REFINE (iter {state.iterations + 1}) — hint: '{hint}'")
            state.iterations += 1

            # FIX: use hint as a *separate* refined query — never touch state.query
            refined_query        = state.query + " " + hint
            original_rewritten   = state.rewritten_query
            state.rewritten_query = refined_query   # only rewritten gets the hint

            state = text_agent(state)
            state = table_agent(state)
            state = metadata_agent(state)
            state = merge_context(state)
            state = synthesizer_agent(state)

            state.rewritten_query = original_rewritten   # restore after loop

            # FIX: if new answer is empty/worse, restore best answer
            new_is_empty = any(p in state.raw_answer.lower() for p in NOT_FOUND_PHRASES)
            if new_is_empty and not best_is_empty:
                print("   ↩️  New answer is worse — restoring best answer.")
                state.raw_answer = state.best_answer
            else:
                state.best_answer = state.raw_answer

        elif verdict == "FAIL":
            print("   ❌ FAIL — appending disclaimer.")
            state.raw_answer  = state.best_answer
            state.raw_answer += "\n\n[⚠️ Critic flagged potential reliability issues. Verify against source documents.]"

        else:
            state.raw_answer = state.best_answer   # always use best

    except Exception as e:
        print(f"   ⚠️  Critic error ({e}), passing through.")

    return state

## 📝 Step 16 — Final Answer Formatter
> Polishes the raw answer into a structured, readable output.

In [16]:
def format_final_answer(state: AgentState) -> AgentState:
    """
    FORMATTER
    ---------
    Cleans and structures the final answer.
    Adds source citations from retrieved chunks.
    """
    print("\n📝  [FORMATTER] Polishing final answer...")

    format_prompt = f"""You are a professional financial report writer.

Given the raw answer below, rewrite it into a clean, structured response.

Rules:
- Keep all numbers and percentages exactly as given.
- Add a one-line summary at the top.
- Cite source type (Text / Table / Metadata) at the end.
- Do NOT add any information not present in the raw answer.
- Max 150 words.

Raw answer:
{state.raw_answer}

Sources used:
- Text chunks: {len(state.text_results)}
- Table chunks: {len(state.table_results)}
- Metadata chunks: {len(state.metadata_results)}

Formatted answer:"""

    try:
        response           = llm.invoke(format_prompt)
        state.final_answer = response.content.strip()
    except Exception as e:
        print(f"   ⚠️  Formatter failed ({e}), using raw answer.")
        state.final_answer = state.raw_answer

    return state

print("✅ Formatter ready")

✅ Formatter ready


## 🎯 Step 17 — Orchestrator
> The master controller that sequences all agents based on the Planner's plan.

In [ ]:
def orchestrator(query: str) -> AgentState:
    """
    ORCHESTRATOR
    ====================
    Fix 1: Router can only select from Planner's agent list (no hallucinated agents).
    Fix 2: Uses critic_agent with best-answer tracking.
    Fix 3: state.query never mutated.
    """
    print("=" * 60)
    print(f"🎯  ORCHESTRATOR — Query: {query}")
    print("=" * 60)

    state = AgentState(query=query)
    state.best_answer = ""

    state = planner_agent(state)
    state = router_agent(state)

    # ── Enforce: Router can only pick from Planner's list ────────────────────
    planner_agents = [a for a in state.plan if a not in ("Critic", "Synthesizer")]
    state.agents_to_call = [a for a in state.agents_to_call if a in planner_agents]
    if not state.agents_to_call:
        state.agents_to_call = planner_agents   # fallback: use full planner list
        print(f"   ↩️  Router returned invalid agents — using Planner's list: {planner_agents}")

    agent_map = {
        "TextAgent":     text_agent,
        "TableAgent":    table_agent,
        "MetadataAgent": metadata_agent,
    }

    for agent_name in state.agents_to_call:
        if agent_name in agent_map:
            state = agent_map[agent_name](state)

    state = merge_context(state)
    state = synthesizer_agent(state)
    state = critic_agent(state)       # ← fixed critic
    state = format_final_answer(state)

    return state



✅ Bug fixes applied — use orchestrator_fixed() instead of orchestrator()


## 🚀 Step 18 — Run the Agentic RAG System

In [18]:
def print_result(state: AgentState):
    print("\n" + "=" * 60)
    print("📋  AGENTIC RAG — FINAL OUTPUT")
    print("=" * 60)
    print(f"Query       : {state.query}")
    print(f"Query Type  : {state.query_type}")
    print(f"Agents Used : {state.agents_to_call}")
    print(f"Iterations  : {state.iterations}")
    print("\n──── Final Answer ────")
    print(state.final_answer)
    print("\n──── Critic Feedback ────")
    print(state.critic_feedback)
    print("=" * 60)

# ── Test Queries ─────────────────────────────────────────────────────────────
queries = [
    "What was the revenue growth of the company?",
    "Which sector does the company belong to?",
    "What is the net income mentioned in the report?",
]

for q in queries:
    result = orchestrator(q)
    print_result(result)
    print("\n")

🎯  ORCHESTRATOR — Query: What was the revenue growth of the company?

🗺️  [PLANNER] Analysing query...
   Query type  : numeric
   Rewritten   : What was the revenue growth of the company? percentage increase year over year r...
   Execution plan: ['TextAgent', 'TableAgent', 'Critic', 'Synthesizer']

🔀  [ROUTER] Deciding agent routing...
   Router decision (LLM): ['TableAgent', 'MetadataAgent']

📊  [TABLE AGENT] Retrieving table chunks...
   Retrieved 5 table chunks
   [0] score=0.3784 | | |  | 2008 | 2007 | 2006 |                                                     ...
   [1] score=0.3502 | | | year ended december 31 ( in millions ) | 2015 | 2014 | 2013 | % (  % ) chang...
   [2] score=0.2011 | | |  | year ended january 1 2006 | year ended january 2 2005 |               |
|...
   [3] score=0.2011 | | |  | year ended january 1 2006 | year ended january 2 2005 |               |
|...
   [4] score=0.2009 | | |  | year ended december 31 2008 ( unaudited ) | year ended december 31 2007 (...

## 🏗️ Architecture Summary

```
USER QUERY
    │
    ▼
┌──────────────┐
│  PLANNER     │  ← Classifies query type, rewrites query, builds execution plan
└──────┬───────┘
       │
    ▼
┌──────────────┐
│  ROUTER      │  ← LLM validates/overrides Planner's agent selection
└──────┬───────┘
       │
   ┌───┴──────────────────────┐
   │                          │
   ▼                          ▼                         ▼
┌──────────┐          ┌──────────────┐         ┌──────────────────┐
│TEXT AGENT│          │ TABLE AGENT  │         │ METADATA AGENT   │
│          │          │              │         │                  │
│HyDE+Dense│          │Dense+Sparse  │         │Dense+Sparse+RRF  │
│+Sparse   │          │+RRF+Rerank   │         │+Rerank           │
│+RRF      │          │+Numeric boost│         │+Structured summary│
│+Rerank   │          └──────┬───────┘         └────────┬─────────┘
└──────┬───┘                 │                          │
       └─────────────────────┼──────────────────────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │ CONTEXT MERGER  │  ← Combines all agent outputs
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │  SYNTHESIZER    │  ← LLM generates answer
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │  CRITIC AGENT   │  ← Checks grounding, completeness
                    │  (ReAct Loop)   │  ← Re-retrieves if REFINE verdict
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │   FORMATTER     │  ← Final clean structured answer
                    └────────┬────────┘
                             │
                             ▼
                       FINAL ANSWER
```


---
## 📏 Step 19 — Evaluation Layer
> RAGAS-style metrics computed **without ground-truth labels** using the LLM as judge.
>
> Four metrics evaluated per query:
> | Metric | What it measures |
> |---|---|
> | **Faithfulness** | Is every claim in the answer supported by the retrieved context? |
> | **Answer Relevancy** | Does the answer actually address the question asked? |
> | **Context Precision** | Are the retrieved chunks relevant to the query? (no noise) |
> | **Completeness** | Does the answer fully resolve the query without gaps? |

### 📐 Step 19b — Evaluation Metric Functions

In [19]:
from dataclasses import dataclass, field as dc_field
from typing import List, Dict

@dataclass
class EvalResult:
    query:             str
    final_answer:      str
    faithfulness:      float = 0.0
    answer_relevancy:  float = 0.0
    context_precision: float = 0.0
    completeness:      float = 0.0
    overall_score:     float = 0.0
    faithfulness_reason:      str = ""
    answer_relevancy_reason:  str = ""
    context_precision_reason: str = ""
    completeness_reason:      str = ""
    agents_used:       List[str] = dc_field(default_factory=list)
    iterations:        int = 0
    query_type:        str = ""


def _parse_score(text: str, key: str) -> tuple:
    """Extract SCORE and REASON for a given key from LLM response."""
    score_m  = re.search(rf"{key}_SCORE:\s*([0-9.]+)", text, re.IGNORECASE)
    reason_m = re.search(rf"{key}_REASON:\s*(.+)",     text, re.IGNORECASE)
    score    = float(score_m.group(1)) if score_m else 0.5
    score    = max(0.0, min(1.0, score))          # clamp to [0,1]
    reason   = reason_m.group(1).strip() if reason_m else "N/A"
    return score, reason


# ── METRIC 1: Faithfulness ───────────────────────────────────────────────────
def evaluate_faithfulness(query: str, answer: str, context: str) -> tuple:
    """
    Score: fraction of answer claims that are grounded in the context.
    1.0 = fully grounded | 0.0 = completely hallucinated
    """
    prompt = f"""You are evaluating a RAG system's answer for faithfulness.

CONTEXT (what was retrieved):
{context[:2000]}

ANSWER (what the system said):
{answer}

Task: Identify every factual claim in the answer.
Count how many are directly supported by the context vs invented.

Score = supported_claims / total_claims  (between 0.0 and 1.0)

Respond EXACTLY:
FAITHFULNESS_SCORE: <0.0 to 1.0>
FAITHFULNESS_REASON: <one sentence>"""

    try:
        r = llm.invoke(prompt)
        return _parse_score(r.content, "FAITHFULNESS")
    except Exception as e:
        return 0.5, f"Eval error: {e}"


# ── METRIC 2: Answer Relevancy ───────────────────────────────────────────────
def evaluate_answer_relevancy(query: str, answer: str) -> tuple:
    """
    Score: how directly and completely the answer addresses the question.
    1.0 = perfectly on-point | 0.0 = completely off-topic
    """
    prompt = f"""You are evaluating whether an answer is relevant to its question.

QUESTION: {query}
ANSWER: {answer}

Score how directly the answer addresses the question.
Penalise answers that are vague, off-topic, or only partially responsive.

Score = relevance (between 0.0 and 1.0)

Respond EXACTLY:
ANSWER_RELEVANCY_SCORE: <0.0 to 1.0>
ANSWER_RELEVANCY_REASON: <one sentence>"""

    try:
        r = llm.invoke(prompt)
        return _parse_score(r.content, "ANSWER_RELEVANCY")
    except Exception as e:
        return 0.5, f"Eval error: {e}"


# ── METRIC 3: Context Precision ──────────────────────────────────────────────
def evaluate_context_precision(query: str, retrieved_docs: List[Dict]) -> tuple:
    """
    Score: fraction of retrieved chunks that are actually relevant to the query.
    1.0 = all chunks useful | 0.0 = all chunks are noise
    """
    if not retrieved_docs:
        return 0.0, "No documents retrieved"

    snippets = ""
    for i, doc in enumerate(retrieved_docs):
        snippets += f"[Chunk {i+1} | {doc.get('type','?')}]: {doc.get('content','')[:200]}\n"

    prompt = f"""You are evaluating context precision in a RAG system.

QUESTION: {query}

RETRIEVED CHUNKS:
{snippets}

For each chunk, decide if it is RELEVANT or IRRELEVANT to answering the question.
Score = relevant_chunks / total_chunks  (between 0.0 and 1.0)

Respond EXACTLY:
CONTEXT_PRECISION_SCORE: <0.0 to 1.0>
CONTEXT_PRECISION_REASON: <one sentence>"""

    try:
        r = llm.invoke(prompt)
        return _parse_score(r.content, "CONTEXT_PRECISION")
    except Exception as e:
        return 0.5, f"Eval error: {e}"


# ── METRIC 4: Completeness ───────────────────────────────────────────────────
def evaluate_completeness(query: str, answer: str, context: str) -> tuple:
    """
    Score: how much of the answerable information was actually used.
    1.0 = nothing left on the table | 0.0 = major gaps
    """
    prompt = f"""You are evaluating whether a RAG answer is complete.

QUESTION: {query}

CONTEXT (available information):
{context[:2000]}

ANSWER (what the system gave):
{answer}

Check: given the context, is there important information the answer missed?
Score = 1.0 if nothing was missed, lower if gaps exist.

Respond EXACTLY:
COMPLETENESS_SCORE: <0.0 to 1.0>
COMPLETENESS_REASON: <one sentence>"""

    try:
        r = llm.invoke(prompt)
        return _parse_score(r.content, "COMPLETENESS")
    except Exception as e:
        return 0.5, f"Eval error: {e}"


print("✅ Evaluation metric functions ready")


✅ Evaluation metric functions ready


### 🚀 Step 19c — Run Full Evaluation

In [21]:
def evaluate_pipeline(queries: List[str], use_fixed=True) -> List[EvalResult]:
    """
    Run the full pipeline on each query, then score with all 4 metrics.
    Set use_fixed=True to use the bug-fixed orchestrator.
    """
    runner  = orchestrator if use_fixed else orchestrator
    results = []

    for query in queries:
        print(f"\n{'='*55}")
        print(f"⚙️   Evaluating: {query}")
        print('='*55)

        # ── Run pipeline ──────────────────────────────────────────────────────
        state = runner(query)

        # ── Collect all retrieved docs ────────────────────────────────────────
        all_docs = state.text_results + state.table_results + state.metadata_results

        # ── Score ─────────────────────────────────────────────────────────────
        print("\n   📐 Scoring metrics...")

        faith_s,  faith_r  = evaluate_faithfulness(query, state.final_answer, state.merged_context)
        relev_s,  relev_r  = evaluate_answer_relevancy(query, state.final_answer)
        prec_s,   prec_r   = evaluate_context_precision(query, all_docs)
        comp_s,   comp_r   = evaluate_completeness(query, state.final_answer, state.merged_context)

        overall = round((faith_s + relev_s + prec_s + comp_s) / 4, 4)

        er = EvalResult(
            query              = query,
            final_answer       = state.final_answer,
            faithfulness       = round(faith_s, 4),
            answer_relevancy   = round(relev_s, 4),
            context_precision  = round(prec_s,  4),
            completeness       = round(comp_s,  4),
            overall_score      = overall,
            faithfulness_reason      = faith_r,
            answer_relevancy_reason  = relev_r,
            context_precision_reason = prec_r,
            completeness_reason      = comp_r,
            agents_used   = state.agents_to_call,
            iterations    = state.iterations,
            query_type    = state.query_type,
        )
        results.append(er)

        print(f"   Faithfulness      : {faith_s:.2f}  — {faith_r}")
        print(f"   Answer Relevancy  : {relev_s:.2f}  — {relev_r}")
        print(f"   Context Precision : {prec_s:.2f}  — {prec_r}")
        print(f"   Completeness      : {comp_s:.2f}  — {comp_r}")
        print(f"   ⭐ Overall         : {overall:.2f}")

    return results


# ── Evaluate on all three queries ─────────────────────────────────────────────
eval_queries = [
    "What was the revenue growth of the company?",
    "Which sector does the company belong to?",
    "What is the net income mentioned in the report?",
]

eval_results = evaluate_pipeline(eval_queries, use_fixed=True)
print("\n✅ Evaluation complete")



⚙️   Evaluating: What was the revenue growth of the company?
🎯  ORCHESTRATOR — Query: What was the revenue growth of the company?

🗺️  [PLANNER] Analysing query...
   Query type  : numeric
   Rewritten   : What was the revenue growth of the company? percentage increase year over year r...
   Execution plan: ['TextAgent', 'TableAgent', 'Critic', 'Synthesizer']

🔀  [ROUTER] Deciding agent routing...
   Router decision (LLM): ['TableAgent', 'MetadataAgent']

📊  [TABLE AGENT] Retrieving table chunks...
   Retrieved 5 table chunks
   [0] score=0.3784 | | |  | 2008 | 2007 | 2006 |                                                     ...
   [1] score=0.3502 | | | year ended december 31 ( in millions ) | 2015 | 2014 | 2013 | % (  % ) chang...
   [2] score=0.2011 | | |  | year ended january 1 2006 | year ended january 2 2005 |               |
|...
   [3] score=0.2011 | | |  | year ended january 1 2006 | year ended january 2 2005 |               |
|...
   [4] score=0.2009 | | |  | year ended dec

### 📊 Step 19d — Evaluation Report

In [22]:
def print_eval_report(results: List[EvalResult]):
    """Pretty-print a full evaluation report with per-query and aggregate scores."""

    METRICS = ["faithfulness", "answer_relevancy", "context_precision", "completeness", "overall_score"]
    LABELS  = ["Faithfulness", "Ans. Relevancy", "Ctx. Precision", "Completeness", "OVERALL"]

    bar = lambda s: ("█" * int(s * 20)).ljust(20) + f" {s:.2f}"

    print("\n" + "═"*70)
    print("               📊  AGENTIC RAG — EVALUATION REPORT")
    print("═"*70)

    for i, er in enumerate(results):
        print(f"\n┌─ Query {i+1}: {er.query[:65]}")
        print(f"│  Type: {er.query_type:<12} Agents: {er.agents_used}  Iterations: {er.iterations}")
        print("│")
        for metric, label in zip(METRICS, LABELS):
            score = getattr(er, metric)
            reason_attr = metric + "_reason"
            reason = getattr(er, reason_attr, "")
            flag = "🟢" if score >= 0.7 else ("🟡" if score >= 0.4 else "🔴")
            print(f"│  {flag} {label:<18} {bar(score)}")
            if reason:
                print(f"│     └ {reason[:70]}")
        print("└" + "─"*69)

    # ── Aggregate ─────────────────────────────────────────────────────────────
    print("\n" + "─"*70)
    print("  📈  AGGREGATE SCORES (mean across all queries)")
    print("─"*70)
    for metric, label in zip(METRICS, LABELS):
        avg = sum(getattr(r, metric) for r in results) / len(results)
        flag = "🟢" if avg >= 0.7 else ("🟡" if avg >= 0.4 else "🔴")
        print(f"  {flag} {label:<18} {bar(avg)}")

    # ── Diagnostic table ──────────────────────────────────────────────────────
    print("\n" + "─"*70)
    print("  🩺  DIAGNOSTIC SUMMARY")
    print("─"*70)

    avg_faith = sum(r.faithfulness      for r in results) / len(results)
    avg_relev = sum(r.answer_relevancy  for r in results) / len(results)
    avg_prec  = sum(r.context_precision for r in results) / len(results)
    avg_comp  = sum(r.completeness      for r in results) / len(results)
    avg_iters = sum(r.iterations        for r in results) / len(results)

    diagnostics = []
    if avg_faith < 0.6:
        diagnostics.append("⚠️  LOW FAITHFULNESS  → Reranker threshold too low; hallucination risk. "
                           "Fix: raise top_k cutoff, add source-grounding instruction to Synthesizer prompt.")
    if avg_relev < 0.6:
        diagnostics.append("⚠️  LOW RELEVANCY     → Planner/Router mismatch; wrong agents called. "
                           "Fix: tighten Router prompt, add query-type examples.")
    if avg_prec < 0.6:
        diagnostics.append("⚠️  LOW CTX PRECISION → Too much retrieval noise. "
                           "Fix: reduce top_k in dense/sparse retrieval, raise rerank_score threshold.")
    if avg_comp < 0.6:
        diagnostics.append("⚠️  LOW COMPLETENESS  → Critic REFINE loop not helping or MAX_ITER too low. "
                           "Fix: increase MAX_ITER to 3, improve HINT extraction in Critic.")
    if avg_iters > 0.8:
        diagnostics.append("⚠️  HIGH ITERATION RATE → Planner query classification may be wrong. "
                           "Fix: add more keyword rules to planner_agent classify block.")

    if not diagnostics:
        print("  ✅  All metrics in healthy range.")
    else:
        for d in diagnostics:
            print(f"  {d}")

    print("\n" + "═"*70)

    # ── Export to DataFrame ───────────────────────────────────────────────────
    rows = []
    for er in results:
        rows.append({
            "Query":             er.query[:50],
            "Type":              er.query_type,
            "Faithfulness":      er.faithfulness,
            "Ans. Relevancy":    er.answer_relevancy,
            "Ctx. Precision":    er.context_precision,
            "Completeness":      er.completeness,
            "Overall":           er.overall_score,
            "Iterations":        er.iterations,
            "Agents":            ", ".join(er.agents_used),
        })
    df = pd.DataFrame(rows)
    print("\n  📋  Full results as DataFrame:")
    return df


eval_df = print_eval_report(eval_results)
eval_df



══════════════════════════════════════════════════════════════════════
               📊  AGENTIC RAG — EVALUATION REPORT
══════════════════════════════════════════════════════════════════════

┌─ Query 1: What was the revenue growth of the company?
│  Type: numeric      Agents: ['TableAgent']  Iterations: 1
│
│  🔴 Faithfulness                            0.00
│     └ The system's answer contains no factual claims directly supported by t
│  🟢 Ans. Relevancy     ████████████████     0.80
│     └ The answer directly addresses the question by providing specific reven
│  🟡 Ctx. Precision     ████████             0.40
│     └ Only 4 out of 13 chunks (Chunks 1, 2, 3, and 4) are relevant to answer
│  🟡 Completeness       ████████████         0.60
│     └ The answer missed the revenue growth information for 2009, which was m
│  🟡 OVERALL            █████████            0.45
└─────────────────────────────────────────────────────────────────────

┌─ Query 2: Which sector does the company belong t

,Query,Type,Faithfulness,Ans. Relevancy,Ctx. Precision,Completeness,Overall,Iterations,Agents
0,What was the revenue growth of the company?,numeric,0.0,0.8,0.40,0.60,0.450,1,TableAgent
1,Which sector does the company belong to?,metadata,1.0,1.0,0.75,0.75,0.875,0,MetadataAgent
2,What is the net income mentioned in the report?,financial,0.5,0.6,0.40,0.50,0.500,1,"TextAgent, TableAgent"
